# 2. Join the federated-learning round

## Goal

This notebook starts the Flower client for **your group only**.

The organiser starts the shared aggregation server. Each group then trains
locally on its own partition and sends model updates—not its CSV rows—to
that server.

**Run the final cell only when the organiser asks all groups to connect.**


## Before you start

Confirm all three statements:

- You completed `01_Inspect_Local_Partition.ipynb`.
- The organiser has confirmed that the Flower server is ready.
- Your group will run the final cell once, and leave it running until it
  reports completion.

The final cell remains busy while the federated rounds are in progress.
This is expected.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import flwr

group_id = os.environ["GROUP_ID"]
workshop_root = Path(os.environ["DIGITAFRICA_WORKSHOP_ROOT"])
data_path = Path(os.environ["CLIENT_DATA_PATH"])
client_program = workshop_root / "app" / "client" / "client.py"
app_root = workshop_root / "app"

if not group_id.startswith("group_"):
    raise RuntimeError(
        f"This notebook requires a workshop group login, got {group_id!r}."
    )
if not data_path.is_file():
    raise FileNotFoundError(f"Local partition is unavailable: {data_path}")
if not client_program.is_file():
    raise FileNotFoundError(f"Workshop client is unavailable: {client_program}")

print("Client readiness check passed.")
print(f"Your group:       {group_id}")
print(f"Local partition: {data_path.name}")
print(f"Flower endpoint: {os.environ['FLOWER_SERVER_ADDRESS']}")
print(f"Flower version:  {flwr.__version__}")


## What will happen after you connect?

For each federated-learning round:

1. the server sends the current shared model to participating groups;
2. each group updates that model using only its local synthetic features;
3. groups return model parameters and summary metrics;
4. the server aggregates the updates into the next shared model.

The server waits until the configured number of groups has connected.


In [ ]:
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

from client.client import load_partition

features, labels = load_partition(
    data_path,
    num_classes=5,
    feature_dim=16,
    seed=42,
)

print(
    f"{group_id} is ready to contribute {len(labels)} local examples "
    f"with {features.shape[1]} synthetic features each."
)
print("No local CSV rows will be sent to the Flower server.")


## Start the client

When the organiser gives the instruction, run the next cell **once**.

You will see connection and training output below the cell. Keep the cell
running. If it reports an error, do not repeatedly restart it; notify the
organiser and include the visible message.


In [ ]:
client_environment = os.environ.copy()
client_environment["PYTHONUNBUFFERED"] = "1"

print(f"Starting the Flower client for {group_id}...")
print("Waiting for the organiser-controlled server and other groups...\n")

process = subprocess.Popen(
    [sys.executable, "-u", str(client_program)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=client_environment,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

exit_code = process.wait()
if exit_code != 0:
    raise RuntimeError(
        f"Flower client for {group_id} stopped with exit code {exit_code}. "
        "Tell the organiser and include the output above."
    )

print(f"\nFederated-learning client for {group_id} completed successfully.")


## After completion

Discuss with the organiser:

- Which information remained local to each group?
- Which model information was shared with the server?
- Why does federated learning still require coordination, validation, and
  an agreed data-governance process?

This workshop is a workflow demonstration, not a validated cassava
disease-classification model.
